## Transaction Level Metrics

In [ ]:
import pandas as pd

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices

In [ ]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

In [ ]:
transaction_data.merge(current_prices, how="left", on="Ticker").fillna(0.0)

## Lot Calculation

In [ ]:
import pandas as pd

from src.backend.input_handler import input_handling

In [ ]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

In [ ]:
# transaction_data = transaction_data.sort_values(by="Transaction Date")

transactions_dict = transaction_data.to_dict(orient='records')
transactions_dict[0:2]

In [ ]:
lots = []

for row in transactions_dict:
    if row.get("Transaction Type") == "BUY":
        lots.append({
            "Portfolio": row.get("Portfolio"),
            "Ticker": row.get("Ticker"),
            "Buy Date": row.get("Transaction Date"),
            "Initial Quantity": row.get("Quantity"),
            "Current Quantity": row.get("Quantity"),
            "Buy Price": row.get("Transaction Price"),
            "Buy Fee": row.get("Fee"),
            "Sell Fee": 0.0,
            "Realized PnL": 0.0,
            "Status": "OPEN",
        })
    elif row.get("Transaction Type") == "SELL":
        quantity_to_sell = row.get("Quantity")

        for lot in lots:
            if quantity_to_sell == 0:
                break

            if lot["Portfolio"] == row.get("Portfolio") and lot["Ticker"] == row.get("Ticker") and lot["Status"] == "OPEN":
                if lot["Current Quantity"] >= quantity_to_sell:
                    quantity_sold = quantity_to_sell
                else:
                    quantity_sold = lot["Current Quantity"]

                pro_rata_ratio = quantity_sold / row.get("Quantity")
                pro_rata_sell_fee = row.get("Fee") * pro_rata_ratio

                pro_rata_buy_fee = lot['Buy Fee'] * (quantity_sold / lot['Initial Quantity'])

                lot["Current Quantity"] -= quantity_sold
                lot["Sell Fee"] += pro_rata_sell_fee

                income = quantity_sold * row.get("Transaction Price")
                cost = quantity_sold * lot["Buy Price"]
                lot["Realized PnL"] += (income - cost - pro_rata_sell_fee - pro_rata_buy_fee)

                quantity_to_sell -= quantity_sold

                if lot["Current Quantity"] == 0:
                    lot["Status"] = "CLOSED"

pd.DataFrame(lots)

## Asset Level Calculation

In [ ]:
import pandas as pd
import numpy as np
from pyxirr import xirr

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices
from src.backend.metrics_calculation import transaction_level_metrics, lot_level_metrics

In [ ]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

transaction_data = transaction_level_metrics(transaction_data, current_prices)

lot_data = lot_level_metrics(transaction_data, current_prices)

In [ ]:
lot_data_temp = lot_data.copy()

lot_data_temp['Cost Of Sold'] = (
    (lot_data_temp["Initial Quantity"] - lot_data_temp["Current Quantity"]) * lot_data_temp["Buy Price"]) + \
    (lot_data_temp["Buy Fee"] * ((lot_data_temp["Initial Quantity"] - lot_data_temp["Current Quantity"]) / lot_data_temp["Initial Quantity"])) + \
    lot_data_temp["Sell Fee"]

lot_data_temp['Cost Of Remaining'] = (
    lot_data_temp["Current Quantity"] * lot_data_temp["Buy Price"]) + \
    (lot_data_temp["Buy Fee"] * (lot_data_temp["Current Quantity"] / lot_data_temp["Initial Quantity"]))

lot_data_temp['Raw Remaining Cost'] = lot_data_temp["Current Quantity"] * lot_data_temp["Buy Price"]

aggs = {
    'Total Fees': ('Total Fees', 'sum'),
    'Total Cost': ('Total Cost', 'sum'),
    'Current Market Value': ('Current Market Value', 'sum'),
    'Realized PnL': ('Realized PnL', 'sum'),
    'Unrealized PnL': ('Unrealized PnL', 'sum'),
    'Current Quantity': ('Current Quantity', 'sum'),
    'Cost Of Remaining': ('Cost Of Remaining', 'sum'),
    'Cost Of Sold': ('Cost Of Sold', 'sum'),
    'Raw Remaining Cost': ('Raw Remaining Cost', 'sum')
}


asset_data = lot_data_temp.groupby(["Portfolio", "Ticker"]).agg(**aggs).reset_index(drop=False)

asset_data["Realized PnL %"] = ((asset_data['Realized PnL'] / asset_data['Cost Of Sold'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data["Unrealized PnL %"] = ((asset_data['Unrealized PnL'] / asset_data['Cost Of Remaining'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data['Total Return %'] = (((asset_data['Realized PnL'] + asset_data['Unrealized PnL']) / asset_data['Total Cost'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data['Average Entry Price'] = (asset_data['Raw Remaining Cost'] / asset_data['Current Quantity'].replace(0, np.nan)).replace(np.nan, 0)

total_mv_per_portfolio = asset_data.groupby('Portfolio')['Current Market Value'].transform('sum')
asset_data['Percentage of Portfolio'] = np.where(total_mv_per_portfolio != 0, (asset_data['Current Market Value'] / total_mv_per_portfolio) * 100, 0)

asset_data

In [ ]:
annualized_gain_pct = []

for row in asset_data.iterrows():
    row = row[1]
    asset_trx = transaction_data[(transaction_data["Portfolio"] == row["Portfolio"]) & (transaction_data["Ticker"] == row["Ticker"])]
    dates = list(asset_trx["Transaction Date"])
    amounts = []

    for _, tx_row in asset_trx.iterrows():
        fee = tx_row.get('Fee', 0.0)
        if tx_row['Transaction Type'].upper() == 'BUY':
            amounts.append(-(tx_row['Quantity'] * tx_row['Transaction Price'] + fee))
        elif tx_row['Transaction Type'].upper() == 'SELL':
            amounts.append((tx_row['Quantity'] * tx_row['Transaction Price'] - fee))
            
    dates.append(pd.Timestamp.today())
    amounts.append(row["Current Market Value"])
    
    try:
        res = xirr(dates, amounts)
        annualized_gain_pct.append(res * 100 if res is not None else np.nan)
    except:
        annualized_gain_pct.append(np.nan)

asset_data['Annualized Gain %'] = annualized_gain_pct
asset_data

## Portfolio Level Calculation

In [ ]:
import pandas as pd
import numpy as np
from pyxirr import xirr

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices
from src.backend.metrics_calculation import transaction_level_metrics, lot_level_metrics, asset_level_metrics

In [ ]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

transaction_data = transaction_level_metrics(transaction_data, current_prices)

lot_data = lot_level_metrics(transaction_data, current_prices)

asset_data = asset_level_metrics(lot_data, transaction_data)